# mouse_type8 動画認識率の検証・モデル比較（Google Colab）

このノートブックは、`mouse_type8` の保存済み走行動画を使い、**同じ実復号 ArUco 盤面登録条件**の上でマウスの認識率を比較検証します。

### 検証できる内容
1. **既存方式の再現評価**: `legacy`（従来色ペア）、`appearance`、`learned_assist`（外観モデル＋小型 CNN 補助）
2. **GPU 実験 A（ローカル互換）**: 既存の小型キーポイント CNN（`BodyKeypointCNN`）を Colab GPU で多ステップ・強拡張学習し、ローカル CPU 互換チェックポイント（`.pt`）を出力
3. **GPU 実験 B（暗い旋回・ブラー対策）**: 色候補・前フレーム追跡に依存せず、AR 正対盤面全体から前後キーポイントを直接推定する `BoardKeypointDetector`（`board_cnn_global`）と、色ペア優先の `hybrid_board_assist` を比較
4. **可視化と目視確認**: 認識率サマリー表、時系列タイムライン（欠測区間の可視化）、比較モンタージュ、差分／飛びフレームの拡大ギャラリー

### 評価ルール（厳守）
- 機体 A / B や前後色の異なる動画は混ぜない。
- 学習動画と評価動画は SHA-256 で独立分割を検証する（同一動画の混入はエラーで停止）。
- 色ペアの実観測率（`observed_given_calibration`）と学習推定込みの被覆率（`coverage_given_calibration`）は分けて評価する。
- 被覆率だけで採用判断せず、静止散らばり・位置/角度の飛び・元フレームへの重畳目視確認を併用する。
- ローカル運用へ反映する場合は、ローカル PC 側で CPU 推論時間と再現性を再検証する。

In [ ]:
# 1. 環境確認（ローカル VS Code / Google Colab 自動判別、GPU / OpenCV aruco / PyTorch）
from pathlib import Path
import platform
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
LOCAL_REPO_DIR = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'tools' / 'analysis' / 'evaluate_body_detectors.py').is_file():
        LOCAL_REPO_DIR = candidate
        break

# ローカル実行時は .tools/vision_python3XX（CPU版PyTorch等）を自動追加
if LOCAL_REPO_DIR is not None:
    vision_dir = LOCAL_REPO_DIR / '.tools' / f'vision_python{sys.version_info.major}{sys.version_info.minor}'
    if vision_dir.is_dir() and str(vision_dir) not in sys.path:
        sys.path.insert(0, str(vision_dir))

print('Execution mode:', 'Google Colab' if IN_COLAB else 'VS Code / Local Python')
print('Python:', sys.version)
print('Platform:', platform.platform())
if LOCAL_REPO_DIR is not None:
    print('Local repo:', LOCAL_REPO_DIR)

import torch
HAS_CUDA = bool(torch.cuda.is_available())
print('PyTorch:', torch.__version__, '| CUDA available:', HAS_CUDA)
if HAS_CUDA:
    print('GPU:', torch.cuda.get_device_name(0))

import cv2
has_aruco = hasattr(cv2, 'aruco') and hasattr(cv2.aruco, 'ArucoDetector')
print('OpenCV:', cv2.__version__, '| ArucoDetector:', has_aruco)
if not has_aruco:
    print('opencv-contrib-python-headless を導入します...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-contrib-python-headless'])
    print('導入完了。必要ならカーネルを再起動して再実行してください。')


In [ ]:
# 2. データソース & 実験パラメータ設定（VS Code ローカル / VS Code Colab 拡張 / ブラウザ Colab 共通）
from datetime import datetime
from pathlib import Path
import zipfile
try:
    from IPython.display import display
except ImportError:
    display = print

# --- ユーザー設定ここから ---
TARGET_MOUSE = 'A'               # 'A' または 'B'

# 実行する実験スイッチ
RUN_BASELINE_EXISTING = True     # バンドル内の既存モデルをそのまま再現評価
RUN_PATCH_CNN_GPU = True         # 既存互換 BodyKeypointCNN を再学習（ローカル CPU 互換保存）
RUN_BOARD_CNN_GPU = True         # AR正対盤面グローバルCNN (BoardKeypointDetector) を学習

# 学習ハイパーパラメータ（GPU 利用時は高精度設定、ローカル CPU 時は軽量設定を自動選択）
DEVICE = 'auto'                  # 'auto' / 'cuda' / 'cpu'
PATCH_CNN_STEPS = 1200 if HAS_CUDA else 400
PATCH_CNN_BATCH_SIZE = 32 if HAS_CUDA else 16
PATCH_CNN_MAX_AUGMENTED = 2048 if HAS_CUDA else 512

BOARD_CNN_PX = 256 if HAS_CUDA else 160
BOARD_CNN_CHANNELS = 24 if HAS_CUDA else 16
BOARD_CNN_DATASET_SIZE = 1024 if HAS_CUDA else 256
BOARD_CNN_STEPS = 1000 if HAS_CUDA else 200
BOARD_CNN_BATCH_SIZE = 32 if HAS_CUDA else 16

# Google Drive 上の配置先（Colab カーネルで /content/ に ZIP が無い場合に使用）
DRIVE_BUNDLE_DIR = Path('/content/drive/MyDrive/mouse_type8_colab_bundles')
DRIVE_REPO_DIR = Path('/content/drive/MyDrive/mouse_type8')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/mouse_type8_colab_results')
# --- ユーザー設定ここまで ---

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
WORK_EXTRACT_DIR = Path('/content/colab_bundles')

if not IN_COLAB and LOCAL_REPO_DIR is not None:
    # VS Code のローカル Python カーネルで実行する場合：ローカルのバンドルをそのまま使用
    REPO_DIR = LOCAL_REPO_DIR
    local_bundles = REPO_DIR / 'tools' / 'logs' / 'pixel_camera' / 'colab_bundles'
    BUNDLE_PATH = local_bundles / f'bundle_{TARGET_MOUSE}'
    if not BUNDLE_PATH.is_dir():
        BUNDLE_PATH = local_bundles / f'bundle_{TARGET_MOUSE}.zip'
    OUTPUT_ROOT = REPO_DIR / 'tools' / 'logs' / 'pixel_camera' / 'colab_results'
    WORK_EXTRACT_DIR = OUTPUT_ROOT / '_extracted'
    TOOLS_ZIP = local_bundles / 'mouse_type8_tools.zip'
else:
    # Google Colab（ブラウザまたは VS Code の Colab 拡張機能リモートカーネル）
    content_bundle = Path(f'/content/bundle_{TARGET_MOUSE}.zip')
    USE_DRIVE = not content_bundle.is_file()
    if IN_COLAB and USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
    REPO_DIR = DRIVE_REPO_DIR if (USE_DRIVE and (DRIVE_REPO_DIR / 'tools' / 'analysis' / 'evaluate_body_detectors.py').is_file()) else Path('/content/mouse_type8')
    BUNDLE_PATH = (DRIVE_BUNDLE_DIR / f'bundle_{TARGET_MOUSE}.zip') if USE_DRIVE else content_bundle
    TOOLS_ZIP = Path('/content/mouse_type8_tools.zip') if Path('/content/mouse_type8_tools.zip').is_file() else (DRIVE_BUNDLE_DIR / 'mouse_type8_tools.zip')
    OUTPUT_ROOT = DRIVE_OUTPUT_ROOT if USE_DRIVE else Path('/content/colab_results')

if not (REPO_DIR / 'tools' / 'analysis' / 'evaluate_body_detectors.py').is_file() and TOOLS_ZIP.is_file():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(TOOLS_ZIP, 'r') as zf:
        zf.extractall(REPO_DIR)
    print('tools ZIP を展開しました:', REPO_DIR)

if not (REPO_DIR / 'tools' / 'analysis' / 'evaluate_body_detectors.py').is_file():
    raise FileNotFoundError(
        f'mouse_type8 リポジトリまたは mouse_type8_tools.zip が見つかりません: {REPO_DIR} / {TOOLS_ZIP}'
    )

for p in (REPO_DIR / 'tools', REPO_DIR / 'tools' / 'analysis'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ZIP の場合は作業ディレクトリへ展開
if BUNDLE_PATH.is_file() and BUNDLE_PATH.suffix.lower() == '.zip':
    WORK_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_PATH, 'r') as zf:
        top_names = sorted({Path(name).parts[0] for name in zf.namelist() if Path(name).parts})
        zf.extractall(WORK_EXTRACT_DIR)
    BUNDLE_DIR = WORK_EXTRACT_DIR / top_names[0] if len(top_names) == 1 else WORK_EXTRACT_DIR
elif BUNDLE_PATH.is_dir():
    BUNDLE_DIR = BUNDLE_PATH
else:
    raise FileNotFoundError(f'バンドルが見つかりません: {BUNDLE_PATH}')

RUN_DIR = OUTPUT_ROOT / f'recognition_{TARGET_MOUSE}_{RUN_TAG}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('TARGET_MOUSE:', TARGET_MOUSE, '| Preset:', 'GPU' if HAS_CUDA else 'CPU (lightweight)')
print('REPO_DIR    :', REPO_DIR)
print('BUNDLE_DIR  :', BUNDLE_DIR)
print('RUN_DIR     :', RUN_DIR)


In [ ]:
# 3. バンドル整合性・機体ID・独立分割（学習/評価リーク）チェック
import json
import pandas as pd
from pixel_camera_detection import file_sha256

bundle_meta = json.loads((BUNDLE_DIR / 'bundle_manifest.json').read_text(encoding='utf-8'))
manifest = json.loads((BUNDLE_DIR / bundle_meta['manifest']).read_text(encoding='utf-8'))

# SHA-256 照合
mismatches = []
for rel, expected_sha in bundle_meta.get('file_sha256', {}).items():
    target = BUNDLE_DIR / rel
    if not target.is_file():
        mismatches.append((rel, 'MISSING'))
    else:
        actual = file_sha256(target).lower()
        if actual != expected_sha.lower():
            mismatches.append((rel, f'SHA mismatch: {actual[:12]} != {expected_sha[:12]}'))
if mismatches:
    raise ValueError(f'バンドル破損を検出しました: {mismatches}')

overlap = set(bundle_meta.get('evaluation_video_sha256', [])) & set(bundle_meta.get('training_video_sha256', []))
if overlap:
    raise ValueError(f'学習動画と評価動画が重複しています: {sorted(overlap)}')

print(f"機体: {bundle_meta['mouse']} | 前色: {bundle_meta['front_colour']} | 後色: {bundle_meta['rear_colour']}")
print('既存モデル:', bundle_meta.get('models'))
print('学習パッチ:', bundle_meta.get('patch_dirs'))
print('SHA-256 整合性・独立分割チェック: OK')

pd.DataFrame([
    dict(
        label=c['label'],
        kind=c['kind'],
        video=c['video'],
        window=c['window'],
        warmup_start=c.get('warmup_start', c['window'][0]),
        stride=c.get('stride', 1),
    )
    for c in manifest['cases']
])


In [ ]:
# 4. 既存方式の再現評価（バンドル内モデルがある場合）
from evaluate_body_detectors import evaluate

existing_app = bundle_meta.get('models', {}).get('appearance_model')
existing_cnn = bundle_meta.get('models', {}).get('cnn_model')
existing_app_path = (BUNDLE_DIR / existing_app) if existing_app else None
existing_cnn_path = (BUNDLE_DIR / existing_cnn) if existing_cnn else None

baseline_report = None
if RUN_BASELINE_EXISTING:
    base_variants = ['legacy']
    if existing_app_path:
        base_variants += ['appearance', 'appearance_refined', 'learned_assist']
    base_manifest = dict(manifest, variants=base_variants)
    baseline_dir = RUN_DIR / 'eval_baseline'
    baseline_report = evaluate(
        base_manifest,
        existing_app_path,
        existing_cnn_path,
        baseline_dir,
        base_dir=BUNDLE_DIR,
        device=DEVICE,
    )
    display_cols = [
        'case', 'variant', 'calibrated_frames', 'aruco_calibration_rate',
        'observed_given_calibration', 'coverage_given_calibration',
        'max_consecutive_missing_frames', 'implausible_position_steps',
        'implausible_heading_steps', 'processing_median_ms',
    ]
    display(pd.DataFrame(baseline_report['results'])[display_cols])


In [ ]:
# 5. Colab GPU 実験 A：既存互換 BodyKeypointCNN の再学習（ローカル CPU 持ち帰り可能）
from body_appearance_model import FRONT_CANON_XY, REAR_CANON_XY
from body_keypoint_cnn import BodyKeypointCNN
from colab_board_detector import load_patches_directory

patch_dirs = [BUNDLE_DIR / rel for rel in bundle_meta.get('patch_dirs', [])]
all_patches_bgr = []
all_patch_sources = []
for pdir in patch_dirs:
    patches, pmeta = load_patches_directory(pdir)
    all_patches_bgr.extend(patches)
    all_patch_sources.extend(pmeta.get('sources', []))

print(f'読み込み学習パッチ数: {len(all_patches_bgr)} 枚')

gpu_patch_cnn_path = None
if RUN_PATCH_CNN_GPU and len(all_patches_bgr) >= 3:
    patch_samples = [(img, FRONT_CANON_XY, REAR_CANON_XY) for img in all_patches_bgr]
    patch_meta = dict(
        mouse=bundle_meta['mouse'],
        front_colour=bundle_meta['front_colour'],
        rear_colour=bundle_meta['rear_colour'],
        sources=all_patch_sources,
    )
    patch_cnn_gpu = BodyKeypointCNN.fit_from_frames(
        patch_samples,
        patch_meta,
        steps=PATCH_CNN_STEPS,
        batch_size=PATCH_CNN_BATCH_SIZE,
        max_augmented=PATCH_CNN_MAX_AUGMENTED,
        robust=True,
        device=DEVICE,
        progress=lambda step, loss: print(f'[PatchCNN] step {step}/{PATCH_CNN_STEPS}: loss={loss:.4f}', flush=True),
    )
    gpu_patch_cnn_path = RUN_DIR / 'models' / 'keypoints_gpu.pt'
    patch_cnn_gpu.save(gpu_patch_cnn_path)
    print('保存完了（CPU互換）:', gpu_patch_cnn_path)
elif RUN_PATCH_CNN_GPU:
    print('学習パッチが3枚未満のため PatchCNN GPU 学習をスキップします。')


In [ ]:
# 6. Colab GPU 実験 B：AR正対盤面グローバル検出器 (BoardKeypointDetector) の学習
# 色ペア候補や前フレーム追跡に依存せず、盤面全体から直接キーポイントを検出して暗い旋回・ブラー時の再捕捉を狙う。
# ※評価動画から背景を作るとデータリークになるため、背景は合成盤面（または学習動画由来のみ）を使用する。
from colab_board_detector import BoardKeypointDetector, make_hybrid_board_detector

board_detector = None
hybrid_board_detector = None
gpu_board_cnn_path = None

if RUN_BOARD_CNN_GPU and len(all_patches_bgr) >= 3:
    first_geom = json.loads((BUNDLE_DIR / manifest['cases'][0]['geometry']).read_text(encoding='utf-8'))
    first_ref = json.loads((BUNDLE_DIR / manifest['cases'][0]['reference']).read_text(encoding='utf-8'))
    board_side_mm = float(first_ref.get('board_side_mm', 360.0))
    nominal_span_mm = float(first_geom.get('nominal_marker_span_mm') or 50.0)

    board_meta = dict(
        mouse=bundle_meta['mouse'],
        front_colour=bundle_meta['front_colour'],
        rear_colour=bundle_meta['rear_colour'],
        sources=all_patch_sources,
    )
    board_detector = BoardKeypointDetector.fit_from_patches(
        all_patches_bgr,
        board_meta,
        backgrounds_bgr=None,
        board_side_mm=board_side_mm,
        nominal_span_mm=nominal_span_mm,
        board_px=BOARD_CNN_PX,
        base_channels=BOARD_CNN_CHANNELS,
        dataset_size=BOARD_CNN_DATASET_SIZE,
        steps=BOARD_CNN_STEPS,
        batch_size=BOARD_CNN_BATCH_SIZE,
        robust=True,
        device=DEVICE,
        progress=lambda step, loss: print(f'[BoardCNN] step {step}/{BOARD_CNN_STEPS}: loss={loss:.4f}', flush=True),
    )
    gpu_board_cnn_path = RUN_DIR / 'models' / 'board_cnn_gpu.pt'
    board_detector.save(gpu_board_cnn_path)
    hybrid_board_detector = make_hybrid_board_detector(board_detector)
    print('保存完了:', gpu_board_cnn_path)
elif RUN_BOARD_CNN_GPU:
    print('学習パッチが3枚未満のため BoardCNN GPU 学習をスキップします。')


In [ ]:
# 7. 全方式の横並び独立ベンチマーク（同じ実復号 ArUco 登録条件）
compare_variants = ['legacy']
extra_detectors = {}
cnn_for_compare = gpu_patch_cnn_path or existing_cnn_path

if existing_app_path is not None:
    compare_variants.append('learned_assist')
if board_detector is not None:
    compare_variants.extend(['board_cnn_global', 'hybrid_board_assist'])
    extra_detectors['board_cnn_global'] = board_detector
    extra_detectors['hybrid_board_assist'] = hybrid_board_detector

compare_manifest = dict(manifest, variants=compare_variants)
compare_dir = RUN_DIR / 'eval_comparison'
compare_report = evaluate(
    compare_manifest,
    existing_app_path,
    cnn_for_compare,
    compare_dir,
    base_dir=BUNDLE_DIR,
    extra_detectors=extra_detectors,
    device=DEVICE,
)

summary_df = pd.DataFrame(compare_report['results'])
summary_df.to_csv(compare_dir / 'summary_table.csv', index=False)
cols = [
    'case', 'kind', 'variant', 'frames', 'calibrated_frames', 'aruco_calibration_rate',
    'observed_given_calibration', 'coverage_given_calibration',
    'missing_given_calibration', 'max_consecutive_missing_frames',
    'implausible_position_steps', 'implausible_heading_steps',
    'processing_median_ms', 'processing_p95_ms',
]
for opt_col in ('midpoint_jitter_rms_mm', 'heading_jitter_rms_deg'):
    if opt_col in summary_df.columns:
        cols.append(opt_col)
display(summary_df[cols])


In [ ]:
# 8. 認識率グラフ & フレーム別検出タイムラインの可視化
import matplotlib.pyplot as plt
import numpy as np

all_frames = pd.read_csv(compare_dir / 'all_frames.csv')

# (A) ケース別・方式別の実観測率 vs 学習補助込み被覆率
cases_list = list(dict.fromkeys(summary_df['case']))
variants_list = list(dict.fromkeys(summary_df['variant']))
fig, axes = plt.subplots(1, len(cases_list), figsize=(5.5 * len(cases_list), 4.2), squeeze=False)
for ax, case_name in zip(axes[0], cases_list):
    sub = summary_df[summary_df['case'] == case_name].set_index('variant').reindex(variants_list)
    x = np.arange(len(variants_list))
    w = 0.36
    ax.bar(x - w / 2, sub['observed_given_calibration'] * 100, width=w, label='Observed (Colour Pair %)')
    ax.bar(x + w / 2, sub['coverage_given_calibration'] * 100, width=w, label='Coverage (Obs + Est %)')
    ax.set_ylim(0, 108)
    ax.set_xticks(x)
    ax.set_xticklabels(variants_list, rotation=20, ha='right')
    ax.set_ylabel('Rate given AR valid [%]')
    ax.set_title(case_name)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(compare_dir / 'recognition_rate_bars.png', dpi=150)
plt.show()

# (B) 時系列タイムライン（0: AR欠測, 1: 機体欠測, 2: 学習推定, 3: 色ペア実観測）
for case_name in cases_list:
    cdf = all_frames[all_frames['case'] == case_name]
    frames_sorted = sorted(cdf['frame'].unique())
    mat = np.zeros((len(variants_list), len(frames_sorted)), dtype=int)
    frame_to_col = {f: i for i, f in enumerate(frames_sorted)}
    for r, v in enumerate(variants_list):
        vdf = cdf[cdf['variant'] == v]
        for _, row in vdf.iterrows():
            c = frame_to_col[row['frame']]
            if not row['calibration_valid']:
                mat[r, c] = 0
            elif row['observed']:
                mat[r, c] = 3
            elif row['estimated']:
                mat[r, c] = 2
            else:
                mat[r, c] = 1
    fig, ax = plt.subplots(figsize=(11, 1.2 + 0.5 * len(variants_list)))
    cmap = plt.matplotlib.colors.ListedColormap(['#444444', '#d62728', '#ff7f0e', '#2ca02c'])
    ax.imshow(mat, aspect='auto', cmap=cmap, vmin=0, vmax=3, interpolation='nearest')
    ax.set_yticks(range(len(variants_list)))
    ax.set_yticklabels(variants_list)
    ax.set_xlabel(f'Evaluated frame index ({frames_sorted[0]} .. {frames_sorted[-1]})')
    ax.set_title(f'{case_name} timeline  [Gray: AR invalid | Red: Missing | Orange: Estimated | Green: Observed]')
    fig.tight_layout()
    fig.savefig(compare_dir / f'timeline_{case_name}.png', dpi=150)
    plt.show()


In [ ]:
# 9. 比較モンタージュ & 差分（従来欠測→回復）フレームの目視確認ギャラリー
import cv2

montage_path = compare_dir / 'comparison_montage.jpg'
if montage_path.is_file():
    img = cv2.cvtColor(cv2.imread(str(montage_path)), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, max(4, img.shape[0] / 120)))
    plt.imshow(img)
    plt.axis('off')
    plt.title('comparison_montage.jpg')
    plt.show()

# legacy が欠測し、他方式が検出したフレームを最大 6 枚抜き出して元画像へ重畳表示
for case in manifest['cases']:
    case_name = case['label']
    cdf = all_frames[(all_frames['case'] == case_name) & (all_frames['calibration_valid'])]
    leg_miss = set(cdf[(cdf['variant'] == 'legacy') & ~(cdf['observed'] | cdf['estimated'])]['frame'])
    other_hit = set(cdf[(cdf['variant'] != 'legacy') & (cdf['observed'] | cdf['estimated'])]['frame'])
    interesting = sorted(leg_miss & other_hit)
    if not interesting:
        print(f'{case_name}: legacy欠測→他方式回復の差分フレームなし')
        continue
    pick = [interesting[i] for i in np.linspace(0, len(interesting) - 1, min(6, len(interesting)), dtype=int)]
    cap = cv2.VideoCapture(str(BUNDLE_DIR / case['video']))
    cap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 0)
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    axes_flat = axes.ravel()
    palette = {'learned_assist': 'orange', 'board_cnn_global': 'magenta', 'hybrid_board_assist': 'cyan'}
    for ax, fidx in zip(axes_flat, pick):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
        ok, frame = cap.read()
        if not ok:
            ax.axis('off')
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rows_f = cdf[(cdf['frame'] == fidx) & (cdf['observed'] | cdf['estimated'])]
        pts_x = pd.concat([rows_f['front_x_px'], rows_f['rear_x_px']]).dropna()
        pts_y = pd.concat([rows_f['front_y_px'], rows_f['rear_y_px']]).dropna()
        cx, cy = float(pts_x.median()), float(pts_y.median())
        x0, x1 = max(0, int(cx - 110)), min(rgb.shape[1], int(cx + 110))
        y0, y1 = max(0, int(cy - 90)), min(rgb.shape[0], int(cy + 90))
        ax.imshow(rgb[y0:y1, x0:x1])
        for _, r in rows_f.iterrows():
            col = palette.get(r['variant'], 'lime')
            ax.plot([r['rear_x_px'] - x0, r['front_x_px'] - x0], [r['rear_y_px'] - y0, r['front_y_px'] - y0], color=col, lw=1.8, label=r['variant'])
            ax.scatter([r['front_x_px'] - x0], [r['front_y_px'] - y0], color=col, s=28, marker='^')
        ax.set_title(f'{case_name} frame {fidx}', fontsize=10)
        ax.axis('off')
        ax.legend(fontsize=7, loc='lower right')
    for ax in axes_flat[len(pick):]:
        ax.axis('off')
    cap.release()
    fig.suptitle(f'{case_name}: legacy missing -> recovered inspection (triangle = front)', fontsize=12)
    fig.tight_layout()
    fig.savefig(compare_dir / f'recovered_gallery_{case_name}.png', dpi=150)
    plt.show()


## 10. ローカル PC への持ち帰り・採用前チェックリスト

出力はすべて `RUN_DIR`（Google Drive 上の新規フォルダー）に保存されます。

- `eval_comparison/report.json`・`summary_table.csv`・`all_frames.csv`: 独立動画での認識率・飛び・処理時間
- `eval_comparison/recognition_rate_bars.png`・`timeline_*.png`・`recovered_gallery_*.png`・`comparison_montage.jpg`: 視覚確認用画像
- `models/keypoints_gpu.pt` (+ `.json`): 既存 `BodyKeypointCNN` と完全互換（CPU テンソル保存済み）
- `models/board_cnn_gpu.pt` (+ `.json`): 盤面グローバル CNN 実験モデル

### ローカル通常解析へ採用する前に確認すること
1. **目視確認**: 回復フレームの前後点が、機体の真の前後マーカー位置に一致しており、背景やバッテリーへの誤認がないこと。
2. **ローカル CPU 再現確認**: ダウンロードしたモデルをローカル PC で `evaluate_body_detectors.py` に通し、推論時間（`processing_median_ms`）と結果の再現性を確認すること。
3. **人手正解による絶対誤差確認**: 色ペア擬似正解からの学習だけでは絶対サブピクセル精度を保証できないため、独立した人手指定点でも誤差を確認すること。
4. **Params / 通常 GUI 設定の分離**: 解析結果だけで `Params/` や通常 GUI の既定値を自動変更せず、機体別の検証記録を残してから反映すること。